In [26]:
import numpy as np
from os import listdir
from os.path import (
    join as join_path,
    isdir
)

from json import load as load_json

In [27]:
def load_json_dict(path):
    try :
        with open(path) as f:
            d = load_json(f)
    except FileNotFoundError:
        return None
    return d
def update_results_dict(comp_name, dname, expno, model_no, results, eval_tracker):
    if not comp_name in results:
        results[comp_name] = {}
    if not dname in results[comp_name]:
        results[comp_name][dname] = {}
    if not expno in results[comp_name][dname]:
        results[comp_name][dname][expno] = {}
    if not model_no in results[comp_name][dname][expno]:
        results[comp_name][dname][expno][model_no] = {}
    results[comp_name][dname][expno][model_no] = eval_tracker
    return results
# def summarize_results(huge_dict):
#     # Structure: {algorithm: {dataset: {exp_number: {model_number: {evaluation_metric: values}}}}}
#     summary = {}

#     for algorithm, datasets in huge_dict.items():
#         for dataset_name, experiments in datasets.items():
#             if dataset_name not in summary:
#                 summary[dataset_name] = {}

#             if algorithm not in summary[dataset_name]:
#                 summary[dataset_name][algorithm] = {}

#             # Collect values per metric
#             metrics_values = {}

#             for exp_num, models in experiments.items():
#                 if models is None:
#                     continue

#                 for model_num, evals in models.items():
#                     if evals is None:
#                         continue
#                     for metric, values in evals.items():
#                         # Skip 'predicted_labels' key
#                         if metric == "predicted_labels":
#                             continue
#                         if values is not None and isinstance(values, list) and values:
#                             metrics_values.setdefault(metric, []).append(values[-1])

#             # Calculate mean and std
#             for metric, values in metrics_values.items():
#                 if isinstance(values, list):
#                     if metric == "no_labeled_pts":
#                         continue
#                     mean = np.mean(values)
#                     std = np.std(values)
#                     summary_str = f"{mean:.2f}±{std:.2f}"
#                 else:
#                     summary_str = "-"
#                 summary[dataset_name][algorithm][metric] = summary_str

#             # Handle case when evals is None or no metrics at all
#             all_metrics = set(metrics_values.keys())
#             if not all_metrics and experiments:
#                 summary[dataset_name][algorithm] = "-"

#     return summary

def summarize_results(huge_dict):
    summary = {}

    for loss_name, labeling_methods in huge_dict.items():
        for labeling_method, datasets in labeling_methods.items():
            for dataset_name, experiments in datasets.items():
                if dataset_name not in summary:
                    summary[dataset_name] = {}

                if loss_name not in summary[dataset_name]:
                    summary[dataset_name][loss_name] = {}

                if labeling_method not in summary[dataset_name][loss_name]:
                    summary[dataset_name][loss_name][labeling_method] = {}

                metrics_values = {}
                labeled_pts = []
                dataset_sizes = []
                cluster_counts = []
                iteration_counts = []

                for exp_num, evals in experiments.items():
                    if evals is None:
                        continue
                    for metric, values in evals.items():
                        if values is None:
                            continue

                        if metric == "no_labeled_pts":
                            if isinstance(values, list) and values:
                                labeled_pts.append(values[-1])
                                iteration_counts.append(len(values))  # Number of iterations
                        elif metric == "dataset_size":
                            if isinstance(values, list) and values:
                                dataset_sizes.append(values[-1])
                            elif isinstance(values, (int, float)):
                                dataset_sizes.append(values)
                        elif metric == "predicted_labels":
                            if isinstance(values, list) and values:
                                unique_labels = np.unique(values)
                                clusters = unique_labels[unique_labels != -1]
                                cluster_counts.append(len(clusters))
                        else:
                            if isinstance(values, list) and values:
                                metrics_values.setdefault(metric, []).append(values[-1])

                # Compute and store labeled ratio and dataset size
                if labeled_pts and dataset_sizes:
                    avg_labeled = np.mean(labeled_pts)
                    std_labeled = np.std(labeled_pts)
                    avg_dataset_size = np.mean(dataset_sizes)
                    labeled_ratio_str = f"{avg_labeled:.2f}±{std_labeled:.2f} /{int(avg_dataset_size)} = {100 * avg_labeled / avg_dataset_size:.2f}%"
                    summary[dataset_name][loss_name][labeling_method]["labeled_ratio"] = labeled_ratio_str
                    summary[dataset_name][loss_name][labeling_method]["dataset_size"] = int(avg_dataset_size)

                # Number of clusters
                if cluster_counts:
                    mean_clusters = np.mean(cluster_counts)
                    std_clusters = np.std(cluster_counts)
                    summary[dataset_name][loss_name][labeling_method]["num_clusters"] = f"{mean_clusters:.2f}±{std_clusters:.2f}"

                # Number of iterations
                if iteration_counts:
                    mean_iter = np.mean(iteration_counts)
                    std_iter = np.std(iteration_counts)
                    summary[dataset_name][loss_name][labeling_method]["num_iterations"] = f"{mean_iter:.2f}±{std_iter:.2f}"

                # Compute other metrics
                for metric, values in metrics_values.items():
                    mean = np.mean(values)
                    std = np.std(values)
                    summary_str = f"{mean:.2f}±{std:.2f}"
                    summary[dataset_name][loss_name][labeling_method][metric] = summary_str

                # Handle case where no metrics are available
                if not metrics_values and experiments:
                    summary[dataset_name][loss_name][labeling_method] = "-"

    return summary


In [28]:
N_MODELS = 5
experiment_namecode = "AE_Sync_corepts_2"
experiment_path = f"/export/share/peters57dm/Verbund/deepsync/results/experiments/{experiment_namecode}"
competitors = listdir(experiment_path)
for lf in competitors:
    _path = join_path(experiment_path, lf)
    if not isdir(_path):
        competitors.remove(lf)

In [29]:
results_dict = {}
for comp_name in competitors:
    datasets_names = listdir(join_path(
        experiment_path, comp_name
    ))
    for dname in datasets_names:
        experiments_numbers = listdir(join_path(
            experiment_path, comp_name, dname
        ))
        for expno in experiments_numbers:
            trackers_path = join_path(
                experiment_path, comp_name, dname, expno
            )
            for model_no in range(N_MODELS):
                evalt_path = join_path(trackers_path, f"results_{model_no}.json")
                eval_tracker = load_json_dict(evalt_path)
                results_dict = update_results_dict(comp_name, dname, expno, model_no, results_dict, eval_tracker)

In [30]:
summary = summarize_results(results_dict)

In [31]:
summary

{'exp_00': {'sync': {'MNIST': {'labeled_ratio': '2465.40±30.00 /2465 = 100.00%',
    'dataset_size': 2465,
    'num_clusters': '10.40±0.49',
    'num_iterations': '1.00±0.00',
    'ari_labeled': '0.85±0.06',
    'ari_total': '0.85±0.06',
    'ami_labeled': '0.92±0.02',
    'ami_total': '0.92±0.02',
    'true_labels': '6.80±1.94'},
   'FMNIST': {'labeled_ratio': '2361.60±20.55 /2362 = 99.98%',
    'dataset_size': 2362,
    'num_clusters': '10.20±1.33',
    'num_iterations': '1.00±0.00',
    'ari_labeled': '0.42±0.09',
    'ari_total': '0.42±0.09',
    'ami_labeled': '0.66±0.05',
    'ami_total': '0.66±0.05',
    'true_labels': '5.20±2.23'},
   'optdigits': {'labeled_ratio': '2477.40±31.31 /2477 = 100.00%',
    'dataset_size': 2477,
    'num_clusters': '11.80±1.17',
    'num_iterations': '1.00±0.00',
    'ari_labeled': '0.85±0.05',
    'ari_total': '0.85±0.05',
    'ami_labeled': '0.93±0.02',
    'ami_total': '0.93±0.02',
    'true_labels': '8.20±0.40'},
   'pendigits': {'labeled_ratio':

In [32]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

def parse_mean_std(s):
    # Dummy version for safety, replace with actual logic
    try:
        if '±' in s:
            mean = float(s.split('±')[0].strip())
            return mean, None
        return float(s), None
    except:
        return None, None

def save_highlighted_summary(summary_dict, save_path):
    all_rows = []
    metric_names = set()
    excluded_metrics = {"predicted_labels"}

    # Step 1: Build the rows for the DataFrame
    for dataset_name, alg_results in summary_dict.items():
        for algorithm_name, metrics in alg_results.items():
            if not isinstance(metrics, dict):
                continue
            row = {
                "Dataset": dataset_name,
                "Algorithm": algorithm_name,
            }
            for metric_name, value in metrics.items():
                if metric_name in excluded_metrics:
                    continue
                row[metric_name] = value
                metric_names.add(metric_name)
            all_rows.append(row)

    # Step 2: Create full DataFrame
    df = pd.DataFrame(all_rows).fillna("-")

    if df.empty:
        print("No data to write.")
        return

    # Step 3: Pivot tables
    def create_metric_table(metric_name):
        return df.pivot(index="Algorithm", columns="Dataset", values=metric_name)

    tables = {}
    for metric in ["ari_total", "ami_total", "ari_labeled", "ami_labeled", 'num_clusters', 'labeled_ratio']:
        if metric in metric_names:
            tables[metric] = create_metric_table(metric)

    # Step 4: Write tables to Excel
    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        start_row = 0
        table_start_rows = {}
        for metric in ["ari_total", "ami_total", "ari_labeled", "ami_labeled", 'num_clusters', 'labeled_ratio']:
            if metric not in tables:
                continue
            table = tables[metric]

            label_text = f"Scores ({metric})"
            # Write the label above the table
            worksheet = writer.book.create_sheet("Results") if "Results" not in writer.book.sheetnames else writer.book["Results"]
            df_label = pd.DataFrame([[label_text]])  # Single-cell DataFrame for label
            df_label.to_excel(writer, sheet_name="Results", startrow=start_row, index=False, header=False)

            # Write the table below the label
            table.to_excel(writer, sheet_name="Results", startrow=start_row + 1)

            table_start_rows[metric] = start_row + 1  # where the actual table starts
            start_row += len(table) + 6  # 1 for label, 1 for header, len for rows, plus spacing

    # Step 5: Add bold labels only
    wb = load_workbook(save_path)
    ws = wb["Results"]
    bold_font = Font(bold=True)

    for metric, table_start in table_start_rows.items():
        label_row = table_start - 1
        ws.cell(row=label_row + 1, column=1, value=f"Scores ({metric})").font = bold_font

    wb.save(save_path)
    print(f"\n Excel with tables saved to: {save_path}")

In [33]:
save_highlighted_summary(summary["exp_00"], join_path(experiment_path, f"results_summary_{experiment_namecode}.xlsx"))


 Excel with tables saved to: /export/share/peters57dm/Verbund/deepsync/results/experiments/AE_Sync_corepts_2/results_summary_AE_Sync_corepts_2.xlsx


In [24]:
np.where(np.diag(core_points_mask) == 1)[0]

array([   4,   12,   13, ..., 6996, 6997, 6998])